# IDAI-780 Capstone — Baseline Demo (Ckpt 2)
**Ryan Khan** · *Chain-of-Thought Reasoning Bias and Search-Agent Retrieval in LLM Financial Fraud Detection*

Runs the whole baseline from this project folder in Google Drive: dataset → classical baselines (train/test) → LLM baselines (Claude oracle and Qwen3 open model) → evaluation.

**Setup:** Runtime → Change runtime type → **GPU**. Add the secret `ANTHROPIC_API_KEY` under 🔑 in the left sidebar and turn on notebook access. Then *Run all*.

## 0. Mount Drive and locate the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob, os, sys, zipfile
PROJECT_DIR = ""   # optional: set the full Drive path to the idai780_baseline folder; leave blank to auto-detect

def locate_project(root="/content/drive/MyDrive", max_depth=5):
    if PROJECT_DIR:
        return PROJECT_DIR
    r = glob.escape(root)
    for d in range(max_depth):                      # unzipped folder already in Drive?
        hits = glob.glob(r + "/*" * d + "/idai780_baseline/src/data.py")
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    for d in range(max_depth):                      # only the zip was uploaded -> extract it
        zips = glob.glob(r + "/*" * d + "/idai780_baseline*.zip")
        if zips:
            parent = os.path.dirname(zips[0])
            print("Extracting", zips[0])
            zipfile.ZipFile(zips[0]).extractall(parent)
            return os.path.join(parent, "idai780_baseline")
    raise FileNotFoundError("idai780_baseline not found in My Drive; set PROJECT_DIR above.")

PROJECT_DIR = locate_project()
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
print("Project folder:", PROJECT_DIR)
!ls

## 1. Environment

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
import anthropic, sklearn, torch, transformers, xgboost
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"anthropic {anthropic.__version__} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__}")

## 2. Dataset
**Source:** Bao et al. (2020), *Journal of Accounting Research* 58(1), public on GitHub. U.S. firm-years from 1990–2014; the label `misstate` comes from SEC AAER enforcement releases; firms are identified only by gvkey.
**Split:** temporal and out of sample. Train on 1991–2001, test on 2003–2008, with a two-year gap. The serial-fraud correction relabels training years of fraud cases that continue into the test period.
**Preprocessing:** median imputation and 1/99% winsorizing, with statistics fit on train only.

In [ ]:
import logging, pandas as pd
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s | %(message)s", datefmt="%H:%M:%S", force=True)
from src import data
df = data.load()
train, test = data.split(df)
print(f"Features: {len(data.RAW_VARS)} raw financial variables + {len(data.RATIO_VARS)} ratios")
pd.DataFrame({
    "years": [f"{train.fyear.min()}-{train.fyear.max()}", f"{test.fyear.min()}-{test.fyear.max()}"],
    "firm_years": [len(train), len(test)],
    "frauds": [int(train.misstate.sum()), int(test.misstate.sum())],
    "fraud_rate_%": [round(100 * train.misstate.mean(), 2), round(100 * test.misstate.mean(), 2)]},
    index=["train", "test"])

## 3. Classical baselines — load → train → test
B0 = random sanity check · B1 = logistic regression on 14 ratios · B2 = XGBoost on 28 raw variables (stand-in for Bao et al.'s RUSBoost, published AUC ≈ 0.71). Add `--quick` to train on a small batch.

In [ ]:
!python -m src.train_classical

## 4a. LLM baseline — Claude oracle
Each sampled test firm-year becomes an anonymized, numbers-only case packet; the model returns a JSON risk score.
Current Claude models take no `temperature`/`top_p`, always run adaptive thinking, and **refuse requests to write out their reasoning** (refusal category `reasoning_extraction`). So for Claude:
- **Direct** = answer only, `effort="low"`.
- **CoT** = an instruction on the order of analysis, answer only, `effort="high"`, with Anthropic's thinking summaries logged.

Visible step-by-step CoT runs on the open model (Step 4b).

In [ ]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
CLAUDE_MODEL = "claude-sonnet-5-5"
EFFORT, COT_EFFORT = "low", "high"     # low | medium | high | xhigh | max
N_CASES = 20                           # balanced: half fraud, half non-fraud
!python -m src.llm_baseline --backend claude --model {CLAUDE_MODEL} --effort {EFFORT} --cot-effort {COT_EFFORT} --max-tokens 16000 --n {N_CASES}

In [ ]:
print(open('results/example_prompt.txt').read())

## 4b. LLM baseline — open model (Qwen3-8B)
Runs in-process with Transformers using greedy decoding, with Qwen's built-in thinking disabled. GPUs under 30 GB load in 4-bit automatically. The first run downloads about 16 GB.

In [ ]:
OPEN_MODEL = "Qwen/Qwen3-8B"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
os.environ["HF_4BIT"] = "1" if gpu_gb < 30 else "0"
print(f"GPU memory {gpu_gb:.0f} GB -> 4-bit = {os.environ['HF_4BIT']}")
!python -m src.llm_baseline --backend hf --model {OPEN_MODEL} --n {N_CASES}

In [ ]:
logs = pd.concat([pd.read_json(f, lines=True) for f in glob.glob("results/llm_*.jsonl")], ignore_index=True)
cols = [c for c in ["model","case_id","label","condition","risk_score","red_flag","warning_type",
                    "thinking_blocks","output_tokens","stop_reason","rationale"] if c in logs.columns]
logs[cols].head(12)

In [ ]:
# Audit trail: Anthropic's summary of Claude's thinking for one CoT case
cl = logs[(logs.backend == "claude") & (logs.condition == "cot")]
if len(cl) and "thinking_summary" in cl:
    print(cl.stop_reason.value_counts().to_string(), "\n")
    print((cl.thinking_summary.dropna().iloc[0] if cl.thinking_summary.notna().any() else "(no summary returned)")[:1500])

## 5. Evaluation on the test set
Full test set: see the Step 3 output (prevalence 0.74%, so a random ranker has AUPRC ≈ 0.007; look at AUC, AUPRC and top-1% precision, not accuracy).
Sampled cases: every baseline is scored on the same balanced cases, plus the paired Δ = risk(CoT) − risk(Direct), a preview of H1.

In [ ]:
!python -m src.evaluate --tau 50

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for (m, c), g in logs.groupby(["model", "condition"]):
    ax[0].scatter(g.label + (0.1 if c == "cot" else -0.1), g.risk_score, alpha=.6, label=f"{m.split('/')[-1]} · {c}")
ax[0].set_xticks([0, 1], ["non-fraud", "fraud"]); ax[0].set_ylabel("LLM risk score")
ax[0].legend(fontsize=7); ax[0].set_title("Risk scores by true label")
w = logs.pivot_table(index=["model", "case_id"], columns="condition", values="risk_score")
d = (w["cot"] - w["direct"]).dropna()
for m, g in d.groupby(level="model"):
    ax[1].hist(g, bins=12, alpha=.6, label=m.split("/")[-1])
ax[1].axvline(0, color="k", lw=.8); ax[1].set_xlabel("Δ = risk(CoT) − risk(Direct)")
ax[1].legend(); ax[1].set_title("Paired condition effect")
plt.tight_layout(); plt.savefig("results/baseline_figures.png", dpi=150); plt.show()
print("Saved to", os.path.join(PROJECT_DIR, "results"))

## 6. (Optional) Package the source code for submission
Writes `idai780_baseline_src.zip` next to the project folder in Drive (code, notebook and README; no data or model files).

In [ ]:
out = os.path.join(os.path.dirname(PROJECT_DIR), "idai780_baseline_src.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("**/*", recursive=True):
        if os.path.isfile(p) and not p.startswith(("data/", "results/")) and "__pycache__" not in p:
            z.write(p, os.path.join("idai780_baseline", p))
print("Wrote", out)